# Minimum-cost network flow: the eight-node lecture example

Factories 1 and 2 supply 3 and 2 truckloads/day, warehouses 3–7 are relay nodes, and customer 8 demands 5 truckloads/day. Flow balance is **outflow − inflow = net supply**.

The data below retain the lecture's $c_{36}=2$ and $u_{36}=4$. The other costs are illustrative; all lower bounds are zero and the other capacities are unlimited. Costs are dollars per truckload, so the objective is dollars/day.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. All required packages are already included, and no external data files are needed.


## Data

`ℓ` denotes the lower bounds ($\ell$); `u` denotes the upper bounds.


In [ ]:
nodes = 1:8
arcs = [(1,3), (2,3), (2,4), (3,5), (3,6), (4,5),
        (5,6), (5,7), (6,7), (6,8), (7,8)]

# Unit costs, in the same order as arcs; c[3,6] = 2 as in the lecture.
c = Dict(zip(arcs, [1, 1, 2, 3, 2, 1, 1, 2, 1, 1, 1]))

# Positive net supply at factories; negative net supply at the customer.
b = [3, 2, 0, 0, 0, 0, 0, -5]

# Arc bounds (truckloads/day).
ℓ = Dict((i,j) => 0 for (i,j) in arcs)
u = Dict((i,j) => Inf for (i,j) in arcs)
u[3,6] = 4


## Model


In [ ]:
using JuMP, HiGHS
import MathOptInterface as MOI

m = Model()

@variable(m, ℓ[i,j] <= x[(i,j) in arcs] <= u[i,j])

@objective(m, Min,
    sum(c[i,j]x[(i,j)] for (i,j) in arcs))

@constraint(m, flowbalance[i in nodes],
    sum(x[(i,j)] for j in nodes if (i,j) in arcs) -
    sum(x[(j,i)] for j in nodes if (j,i) in arcs)
    == b[i])

set_optimizer(m, HiGHS.Optimizer)
set_silent(m)
optimize!(m)


## Solution


In [ ]:
println("Status: ", termination_status(m))
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end

println("Minimum cost = ", round(objective_value(m), digits=1), " dollars/day")
for (i,j) in arcs
    if value(x[(i,j)]) > 0.00001
        println("Flow from node ", i, " to node ", j, " = ",
            round(value(x[(i,j)]), digits=1), " truckloads/day")
    end
end
